# Task 0 — Preparazione dei dati (Kaggle, solo CPU)

Questo notebook esegue gli script della repo, senza logica propria:
1. pulisce le maschere e calcola i box (`scripts/prepare_data.py`);
2. crea lo split train/val/test e i prompt fissi per val e test (`scripts/make_split.py`);
3. mostra i controlli visivi da fare a occhio.

**Session options:** *Accelerator* **None** (la GPU non serve, così non consumi quota) · *Internet* **On** · *Persistence* **Files only**.

## 1. Codice della repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<tuo-utente>/sam-peft-kvasir.git"   # <-- da modificare
WORK = Path("/kaggle/working")
REPO = WORK / "sam-peft-kvasir"

# se hai caricato la repo come dataset Kaggle la usa da lì, altrimenti fa git clone
uploaded = next(Path("/kaggle/input").rglob("pyproject.toml"), None) if Path("/kaggle/input").exists() else None
if uploaded is not None and not REPO.exists():
    !cp -r {uploaded.parent} {REPO}
elif not REPO.exists():
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q

!pip install -q -e "{REPO}[dev]"
%cd {REPO}

## 2. Dati grezzi

In [ ]:
def find_kvasir_root():
    for base in [Path("/kaggle/input"), WORK / "data"]:
        if base.exists():
            for p in base.rglob("images"):
                if (p.parent / "masks").is_dir():
                    return p.parent
    return None

RAW = find_kvasir_root()
if RAW is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q -L --no-check-certificate https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/ && rm /kaggle/working/data/kvasir-seg.zip
    RAW = find_kvasir_root()
assert RAW is not None, "Kvasir-SEG non trovato"
print(RAW, len(list((RAW / "images").glob("*.jpg"))), "immagini")

## 3. Test
Se un test fallisce, fermati qui: gli script sotto darebbero risultati non affidabili.

In [ ]:
!python -m pytest -q

## 4. Pulizia delle maschere e metadati
Leggi soprattutto `removed_*` (quanto rumore è stato tolto) e il confronto `big_box_raw_union` contro `big_box_clean`.

In [ ]:
PROCESSED = WORK / "processed"
!python scripts/prepare_data.py --raw-root {RAW} --out {PROCESSED}

## 5. Controllo visivo della pulizia
Le immagini con più pixel rimossi: nella terza colonna (in rosso) c'è ciò che è stato tolto.
Deve essere solo rumore sparso, mai un pezzo di polipo.

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sampeft.data.masks import load_image, load_raw_mask

meta = json.loads((PROCESSED / "meta.json").read_text())

def show_cleaning(img_id):
    img = load_image(RAW / "images" / f"{img_id}.jpg")
    raw = load_raw_mask(RAW / "masks" / f"{img_id}.jpg")
    clean = cv2.imread(str(PROCESSED / "masks_clean" / f"{img_id}.png"), 0) > 0
    removed = raw & ~clean
    over = np.dstack([clean, clean, clean]).astype(np.uint8) * 255
    over[removed] = [255, 0, 0]
    over = cv2.dilate(over, np.ones((5, 5), np.uint8))       # ingrandisce i pixel rossi per vederli
    m = meta[img_id]
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(img); ax[1].imshow(raw, cmap="gray"); ax[2].imshow(over)
    ax[0].set_title(img_id); ax[1].set_title("raw (GT ufficiale)")
    ax[2].set_title(f"clean, rosso = rimosso ({m['removed_pixels']} px, {m['removed_components']} comp.)")
    for a in ax: a.axis("off")
    plt.show()

worst = sorted(meta, key=lambda i: meta[i]["removed_pixels"], reverse=True)[:4]
most_polyps = max(meta, key=lambda i: meta[i]["n_polyps"])
for img_id in dict.fromkeys(worst + [most_polyps]):
    show_cleaning(img_id)

## 6. Split e prompt di val/test
Lo split è salvato in `data/splits/` dentro la repo, perché va versionato con git.

In [ ]:
!python scripts/make_split.py --raw-root {RAW} --processed {PROCESSED} --out data/splits

## 7. Controllo visivo dei quasi-duplicati
Per ogni gruppo trovato: sono davvero frame quasi uguali? Se vedi immagini diverse, la soglia è troppo alta
(rilancia la cella 6 con `--dup-threshold 2`).

In [ ]:
dups = json.loads(Path("data/splits/duplicates.json").read_text())
print(len(dups), "gruppi")
for g in dups[:8]:
    fig, ax = plt.subplots(1, len(g["ids"]), figsize=(4 * len(g["ids"]), 3))
    for a, i in zip(np.atleast_1d(ax), g["ids"]):
        a.imshow(load_image(RAW / "images" / f"{i}.jpg")); a.set_title(i[:12]); a.axis("off")
    fig.suptitle(f"distanza max: {g['max_distance']} bit su 64")
    plt.show()

## 8. Salvataggio
- `data/splits/*.json` → scaricali (pannello Output) e fai commit nella repo.
- `/kaggle/working/processed/` → *Save Version*, poi dall'output della versione **New Dataset** con nome `kvasir-seg-processed`. Dai prossimi notebook lo aggiungi come input.

In [ ]:
!ls -la data/splits {PROCESSED}
!cd /kaggle/working && zip -qr splits.zip sam-peft-kvasir/data/splits && ls -la splits.zip

## Da annotare (va nel README)
1. `stats.json`: pixel rimossi in totale, `big_box_raw_union` vs `big_box_clean`.
2. Le maschere pulite perdono mai pezzi di polipo? (sezione 5)
3. Quanti gruppi di quasi-duplicati, e sono veri duplicati? (sezione 7)
4. Conteggi dello split: multi-polipo e box enormi sono distribuiti in modo bilanciato?